# Influence des médailles remportées par la France aux Jeux Olympiques sur le nombre le licenciés sportifs en France.

_Projet Python pour la Data Science — Melissa MIGAN, Camille PEYTHIEUX-TALDIR, Romane PLUQUET — 2025._

## Introduction

Les Jeux Olympiques de 2024 se sont tenus du 26 juillet au 11 août 2024 à Paris, en Seine-Saint-Denis, à Marseille et à Tahiti. Ils ont été présentés comme une réussite sportive pour la France : la délégation française y a obtenu son record de médailles depuis 1900, et l'athlète le plus titré des Jeux de Paris est le nageur français Léon Marchand (4 médailles d'or). De plus, cette édition a permis de mettre en lumière et de médiatiser des sports plus méconnus en France, comme le tennis de table grâce aux frères Lebrun. 

Le 23 juin 2023, le Président de la République française Emmanuel Macron s'exprimait sur France Télévision. Il y développait l'idée suivante : "_En matière sportive, qu’est-ce que ça veut dire ? Grâce à nos JO, [...] accompagner nos clubs, nos associations, évidemment, toutes les fédérations sportives qui structurent le sport en France[...]. L'objectif de tout ça, c'est faire quoi ? De la France, cette nation sportive. De plus en plus de jeunes et d'adultes pratiqueront le sport en vue des J.O._" Emmanuel Macron insiste alors sur l'idée suivante : les Jeux Olympiques de Paris 2024 auraient également pour but de dynamiser la pratique sportive en France.

Ainsi, nous pouvons nous poser la question suivante : les Jeux Olympiques, et plus précisément les victoires sportives aux Jeux Olympiques, ont-ils une influence sur la pratique sportive ? Le fait de remporter des médailles olympiques incite-t-il à la pratique sportive ?

Ici, nous considèrerons que le nombre de médailles olympiques remportées représente les victoires sportives aux Jeux Olympiques. De plus, nous mesurerons la pratique sportive en France par le nombre de licenciés sportifs.

### Sommaire

- **Introduction**

- **I. Création et exploration de la base de données principale**
    - A. Récupération des données
        - _1. Médailles françaises aux Jeux Olympiques_
        - _2. Licenciés sportifs en France_
        - _3. Population départementale en France_
    - B. Jointure des tables
    - C. Contrôle de la qualité des données
        - _1. Structure de la base_
        - _2. Valeurs manquantes_
        - _3. Cohérence des valeurs_
        - _4. Unicité des observations_
    - D. Export de la base finale

- **II. Visualisations et statistiques descriptives**
    - A. Exploration "naïve" de la base
        - _1. Etude des licenciés sportifs en France_
        - _2. Etude des médailles olympiques remportées par la France_
    - B. Medailles olympiques et licenciés sportifs
    - C. Evolution du nombre de licenciés : analyse socio-démographique croisée avec le profil des athlètes
        - _1. Une diffusion ou une concentration de la pratique sportive ?_
        - _2. Les jeunes sont-ils plus inspirés par les athlètes olympiques ?_
        - _3. Les médaillées des Jeux sont-elles un exemple de pratique sportive pour les femmes ?_
    
- **III. Modélisation : effet des médailles olympiques sur les licenciés**
    - A. Objectif
    - B. Enrichissement du jeu de données et variables explicatives
        - *1. Variable dépendante : nombre total de licenciés*
        - *2. Structure démographique des licenciés*
        - *3. Dispersion géographique de la pratique*
        - *4. Variables liées aux Jeux Olympiques*
        - *5. Sanity checks (cohérence et reproductibilité)*
    - C. Modèle économétrique principal : croissance post-JO (panel avec effets fixes)
    - D. Diagnostics / “preuves” complémentaires
        - *1. Preuve 1 — Ablation prédictive (inertie vs médailles)*
        - *2. Preuve 2 — Poids de l’inertie (coefficients Ridge)*
        - *3. Preuve 3 — Les médailles apportent-elles une information marginale dans le modèle économétrique ?*
        - *4. Portée des résultats et limites*
    - E. Conclusion de la modélisation

- **Conclusion**

In [ ]:
%pip install -r requirements.txt

## I. Création et exploration de la base de données principale

Dans cette partie, l'objectif est d'importer et de travailler les différentes bases de données et de les joindre en une base de données exploitable. Après travail et nettoyage des données brutes (en accès public), nous utilisons trois bases de données :

| Nom de la base   | Description                                                                 | Source     | Mode d'extraction |
|-----------------|------------------------------------------------------------------------------|------------|-------------------|
| `data_medailles`  | Nombre de médailles reportées par la France aux Jeux Olympiques par sport et année (2016-2024). | Wikipedia ([lien](https://fr.wikipedia.org/wiki/France_aux_Jeux_olympiques)).  | Web scraping      |
| `data_licences`   | Nombre de licenciés sportifs en France par fédération, sexe, âge et département (2016-2024).       | Injep, Institut national de la jeunesse et de l'éducation populaire ([lien](https://injep.fr/donnee/recensement-des-licences-et-clubs-sportifs-2024/)).          | CSV, Parquet      |
| `data_pop`        | Population départementale en France, recensements de 2016 et 2022.          | Insee  ([lien](https://www.insee.fr/fr/statistiques/3698339)).    | API               |

Ces tables regroupent les données suivantes :
- La table `data_medailles` rassemble les médailles obtenues par la France aux Jeux Olympiques entre 2016 et 2024, c'est-à-dire aux JO de 2016, 2020 (qui ont eu lieu en 2021 à cause du Covid) et de 2024. 
- La table `data_licences` recense les effectif de licenciés par an entre 2016 et 2024. Elle présente également des effectifs par tranches d'âge et par genre.
- La table `data_pop` contient les populations départementales recensées en 2016 et en 2022. Elle nous permet de mener une étude des effectifs de licenciés par département, relativement à la population de ces derniers. Cette table n'étant utilisée que dans un unique graphique (dans un but de pondération), elle ne sera pas incluse dans notre base de données principale.

On importe les packages pour traiter les données, et les fonctions utilisées.

In [ ]:
# Modules
#import os
import pandas as pd
import numpy as np
import pyarrow as pa
from pathlib import Path


In [ ]:

# Pipes
from traitement.prep import (
    gel_tableau_medailles,
    nettoyer_base,
    fusionner_bases,
    reorganiser_colonnes,
    normalisation_unicode,
    code_sport,
    code_dep,
    renommer_colonnes, 
    gel_parquet,
    tableau_ratios_nr,
    melodi_extraction,
    code_dep_pop,
    clean_population,
    gel_population
)

### A. Récupération des données

#### 1. Médailles françaises aux Jeux Olympiques

Nous avons scrapé la page Wikipedia ["_France aux Jeux Olympiques_"](https://fr.wikipedia.org/wiki/France_aux_Jeux_olympiques) afin d'obtenir les tableaux des médailles (or, argent bronze) obtenues par la France lors des Jeux Olympiques, à l'aide de la fonction `tableau_scraper`.

In [ ]:
a_figer = [["or", "M.C3.A9dailles_d.27or_3"],
           ["argent", "M.C3.A9dailles_d.27argent"],
           ["bronze", "M.C3.A9dailles_de_bronze"]]

for duo in a_figer:
    gel_tableau_medailles(duo[0], duo[1])

 Nous avons ensuite gelé les bases scrapées dans un souci de reproductibilité, au cas où la page Wikipédia soit modifiée. Nous avons ainsi obtenu trois tables :
- `data_or` : renseignant le nombre de médailles d'or obtenues par la France aux Jeux Olympiques, par sport et par année,
- `data_argent` : renseignant le nombre de médailles d'argent obtenues par la France aux Jeux Olympiques, par sport et par année,
- `data_bronze` : renseignant le nombre de médailles de bronze obtenues par la France aux Jeux Olympiques, par sport et par année.

In [ ]:
data_or = pd.read_csv(f"data/data_brut/data_or_jo.csv")
data_argent = pd.read_csv(f"data/data_brut/data_argent_jo.csv")
data_bronze = pd.read_csv(f"data/data_brut/data_bronze_jo.csv")

display(data_or.head())
display(data_argent.head())
display(data_bronze.head())

Nous avons ensuite nettoyé ces tables :
- nous avons supprimé les lignes et colonnes vides, ainsi que la colonne `Place`, qui ne servait que lors du scraping (pour être sûr de bien scraper toutes les lignes) ;
- nous avons retiré les années hors de notre période d'analyse (2016-2024) ;
- nous avons supprimé la colonne de total des médailles.

In [ ]:
data_or_clean = nettoyer_base(data_or)
data_argent_clean = nettoyer_base(data_argent)
data_bronze_clean = nettoyer_base(data_bronze)

display(data_or_clean.head())
display(data_argent_clean.head())
display(data_bronze_clean.head())

Suite à cela, nous avons joint ces trois tables afin d'obtenir la table `data_medailles_jo`, renseignant le nombre de médailles (toutes couleurs confondues) obtenues par la France aux Jeux Olympiques de 2016, 2020 et 2024.

Nous y avons ajouté trois variables :
- `code_sport`, un code permettant plus tard la jointure avec la table des licenciés sportifs en France, dont la modalité DIV correspond à des sports olympiques n'ayant pas de fédération associée dans notre base de licences, 
- `total_medailles_2020`, une somme de toutes les médailles remportées lors des Jeux Olympiques de 2020,
- `total_medailles_2024`, une somme de toutes les médailles remportées lors des Jeux Olympiques de 2024.

In [ ]:
data_medailles = fusionner_bases(data_or, data_argent, data_bronze)
display(data_medailles.sample(5))

#### 2. Licenciés sportifs en France

Nous avons exploité les données de licenciés sportifs en France fournies par l'Injep, l'Institut national de la Jeunesse et de l'Education populaire. Les données se présentent sous forme brute comme un fichier CSV par an. Cependant, ces fichiers étant trop lourds pour être importés directement dans Git, nous optons pour une gestion des données par fichiers au format parquet. 


##### a. Construction de la base 

Notre but ici est de contruire une base de données au format long. Chaque base disposant déjà d'une colonne année, nous devons alors les concaténer pour obtenir la base au format désiré. Pour que l'opération se déroule correctement, nous réorganisons dans un premier temps les colonnes de chacune des bases, de sorte que chacune ait les mêmes colonnes dans le même ordre.

In [ ]:
liste_fichiers = ["Lics_2016_semidef.parquet",
                  "Lics_2017_semidef.parquet", 
                  "Lics_2018_semidef.parquet",
                  "Lics_2019_def.parquet",
                  "Lics_2020_def.parquet",
                  "Lics_2021_def.parquet",
                  "Lics_2022_def.parquet",
                  "Lics_2023_semidef.parquet",
                  "Lics_2024_semidef.parquet"]

data_licences = reorganiser_colonnes(liste_fichiers)
data_licences = pa.concat_tables(data_licences)

Afin d'éviter les problèmes de sélection de données, car nous disposons de variables dont les modalités sont textuelles, nous normalisons tous les caractères avec la norme unicode. Nous ajoutons ensuite plusieurs variables permettant un niveau d'analyse plus général que celui très fin proposé par la base de données ainsi qu'une harmonisation entre les différentes bases utilisées : 
- `code_sport` : catégorise les fédérations selon le sport pratiqué. Nous choisissons de catégoriser en "divers" (`code_sport` : DIV) les fédérations qui pratiquent un sport non-olympique. Ce code est le même que celui ajouté à la table des médailles. 
- `code_dep` : indique le département par son numéro seulement. 

In [ ]:
data_licences = normalisation_unicode(data_licences)
data_licences = code_sport(data_licences)
data_licences = code_dep(data_licences, "Département")

Nous renommons ensuite les colonnes pour avoir des noms de variable sans majuscules ni accents. Nous ne gardons que les colonnes qui nous seront utiles pour la suite, à savoir celles concernant le nom de la fédération, l'année de recensement des licences, le sexe des licencié.e.s, les tranches d'âge (âge, tranches fines et grandes tranches), le nombre de licenciés annuel, le code sport et le code département. Finalement, nous gelons la table dans un fichier parquet pour la réutiliser par la suite telle que construite ici. 

In [ ]:
data_licences = renommer_colonnes(data_licences)

data_licences = data_licences[
    ["federation",
    "annee",
    "sexe",
    "age",
    "tranche_age",
    "grande_tranche_age",
    "licences_annuelles",
    "code_sport",
    "code_dep",
    "region"]
]

gel_parquet(data_licences, "data_licences.parquet")

display(data_licences.sample(5))
display(data_licences["federation"].describe())

Nous obtenons une base de données comprenant plus de 7,35 millions de lignes. 118 fédérations sportives y sont recensées, et ce sur neuf années, de 2016 à 2024. L'année indiquée dans la base ne correspond pas nécessairement à l'année d'inscription dans la licence. D'après la documentation des fichiers d'effectifs de licences, "les données du millésime N correspondent à la saison sportive N-1/N ou à l'année civile N selon le fonctionnement des fédérations". Ainsi, les données de 2016 peuvent correspondre à des inscriptions ayant eu lieu en 2015 ou en 2016, une saison sportive commençant classiquement en septembre et se terminant début juillet. 

La variable `code_sport` nous permet de catégoriser facilement les fédérations selon le sport pratiqué. Nous choisissons la modalité "DIV" pour les sport non-olympiques. Ainsi, 33 sports olympiques sont présents dans notre base de données. 

Nous disposons de plusieurs variables permettant de distinguer les licenciés selon des critères socio-démographiques :
- trois variables d'âge :  
    - la variable `age` qui donne l'âge précis des licenciés ;
    - la variable `tranche_age` qui répertorie les licenciés selon 18 tranches d'âges "fines", par exemple ceux ayant de 10 à 14 ans ;
    - la variable `grande_tranche_age` qui classe les licenciés selon cinq tranches d'âge plus larges, comme par exemple la catégorie Adultes (21-55 ans) ;
- la variable `sexe`, qui présente deux modalités (H ou F) ;
- la variable `code_dep`, qui renseigne le département dans lequel sont enregistrés les licenciés.

Finalement, notre variable d'intérêt est la variable `licences_annuelles`, que nous pouvons agréger selon toutes les variables présentes dans la base, en prenant les précautions nécessaires, détaillées dans la partie suivante. 

##### b. Précautions : comparabilité dans le temps

Nos données de licences proviennent de fichiers distincts pour chaque année recensée. Ces fichiers sont de deux types :
- `semidef` pour les années 2016 à 2018 et 2023 à 2024, qui n'ont pas encore été "géocodées",
- `def` pour les années 2019 à 2022, qui sont "géocodées".

La documentation de ces données affirme que les données sont comparables dans le temps à condition d'être agrégées par fédération. Ainsi, comme notre code sport est directement dérivé des fédérations, l'agrégation selon la variable `code_sport` ne posera pas de problème de compabilité. Cependant, il est précisé que les "données par sexe, et/ou par âge, et/ou par département/région ne doivent pas être comparées dans le temps directement". Il faut dans ce cas là bien prendre en compte les effectifs non répartis (NR), c'est-à-dire qui n'ont pas pu être classés selon un département, une catégorie d'âge ou de genre, afin d'obtenir des résultats comparables dans le temps. 

Pour d'avoir une idée de l'ampleur de la non répartition géographique des effectifs de licences dans les données, nous calculons les ratios d'effectifs de licences géographiquement non répartis sur la totalité des effectifs de licences par an, ainsi que dans la base regroupant toutes les années. 

In [ ]:
display(tableau_ratios_nr(data_licences, "code_dep"))

On constate alors que la proportion de licences non géographiquement réparties est bien plus importante pour les premiers fichiers `semidef`, c'est-à-dire de 2016 à 2018, alors que par la suite, cette proportion n'excède pas les 1%. Cependant, la présence d'effectifs géographiquement non répartis nous posera problème par la suite, lors de la représentation de ces effectifs sous forme de cartes. De ce fait, nous ne pourrons pas les inclure dans nos représentations cartographiques. Cette décision nous paraît raisonnable dans la mesure où une faible proportion des effectifs est géographiquement non répartie après 2018. Toutefois, il ne faudra pas oublier que de 2016 à 2018, autour de 5% des effectifs de licences ne sont géographiquement pas attribués.

Nous montrons ensuite que, comme le suggère la précision dans la documentation sur le fait que certains fichiers soient "géocodés" ou non, la non répartition géographique est la plus importante dans notre jeu de données par rapport aux autres variables socio-démographiques. Nous calculons alors les ratio de non répartis selon ces autres variables. 

In [ ]:
display(tableau_ratios_nr(data_licences, "sexe"))
display(tableau_ratios_nr(data_licences, "age"))
display(tableau_ratios_nr(data_licences, "tranche_age"))
display(tableau_ratios_nr(data_licences, "grande_tranche_age"))

En ce qui concerne l'âge et le sexe, on constate que la non répartition est globalement moins importante, mais toujours assez marquée de 2016 à 2018. La non répartition pour les variables d'âge en tranches est strictement égale à celle de la variable d'âge, puisque ces dernières découlent directement de la première. La non répartition en terme d'âge n'excède pas les 2% par an, et tend vers de très faibles valeurs à partir de 2019 (< 0,31%). La non répartition par sexe, elle, n'excède pas les 2,43% et est nulle de 2020 à 2024. Ainsi, bien que ce phénomène soit moins important pour l'âge et le sexe, nous le prendrons en compte dans nos analyses par âge et par sexe. 

#### 3. Population départementale en France

Dans l'optique de mener une analyse des effectifs de licenciés par départements, proportionnellement à leur population, nous récupérons les données de population départementale en France grâce à l'API Melodi de l'INSEE. Nous choisissons d'utiliser le jeu de données de la population de municipale et non de la population de référence, car la population municipale est comptabilisée tous les ans, tout comme nos données de licences. Ce choix permet une cohérence temporelle entre nos données. Par ailleurs, d'après l'INSEE, la population municipale est "désormais [...] la notion de population la plus utilisée en statistiques".

La documentation de l'API fournit directement le code permettant d'extraire les données voulues, que nous reprenons dans la fonction permettant l'extraction des données. L'URL d'extraction a été obtenu par visualisation de la base de données au niveau de précision désiré, c'est-à-dire au niveau départemental ; et permet de sélectionner directement les années d'intérêt (2016 à 2023). 

La base de données extraite comporte 800 lignes : la population pour les 96 départements de France métropolitaines et quatre départements d'Outre-mer (Guadeloupe, Martinique, Guyane, La Réunion), sur huit années. Les données n'étant pas encore disponibles pour l'année 2024, nous utiliserons celles de 2023 pour des analyses relatives à la population en 2024.

In [ ]:
url_api = "https://api.insee.fr/melodi/data/DS_POPULATIONS_HISTORIQUES?TIME_PERIOD=2016&TIME_PERIOD=2017&TIME_PERIOD=2018&TIME_PERIOD=2019&TIME_PERIOD=2020&TIME_PERIOD=2021&TIME_PERIOD=2022&TIME_PERIOD=2023&GEO=DEP"
data_pop = melodi_extraction(url_api)
display(data_pop.sample(5))

Nous ne sélectionons que les variables utiles pour notre analyse, c'est-à-dire celles correspondant au département, à l'année et à la population comptablisée dans le département, puisque les variables de fréquence de mesure et de type de mesure sont unimodales. Nous renommons également les colonnes avec le même format que pour les bases présentées précédemment. Finalement, nous ajoutons le même code département que dans les autres bases, à partir de la variable de département. 

In [ ]:
display(data_pop["FREQ"].unique())
display(data_pop["POPREF_MEASURE"].unique())

In [ ]:
data_pop = data_pop[["GEO", "TIME_PERIOD", "OBS_VALUE_NIVEAU"]]
data_pop.columns = ["departement", "annee", "population"]
data_pop = code_dep_pop(data_pop, "departement")

Nous nettoyons ensuite cette base, en forçant les types des variables (année en entiers, département en variables textuelles), en supprimant les départements non cartographiables et en les triant par ordre croissant. Nous gelons ensuite les données dans un fichier CSV. 

In [ ]:
data_pop_clean = clean_population(data_pop)
gel_population(data_pop_clean)
data_pop_clean.sample(5)

### B. Jointure des tables

Nous avons joint les tables médailles et licences pour pouvoir étudier en détail l'effet de remporter des médailles aux Jeux Olympiques sur l'évolution du nombre de licenciés sportifs. Nous avons joint par la gauche en utilisant la clé `code_sport` pour ne pas démultiplier le nombre de lignes dans notre DataFrame. La jointure par la gauche est possible car le `code_sport` est un identifiant unique dans la base des médailles. 

In [ ]:
data_complet = pd.merge(data_licences, data_medailles, how='left', on="code_sport")
data_complet.sample(5)

### C. Contrôle de la qualité des données

#### 1. Structure de la base

In [ ]:
data_complet.shape

In [ ]:
data_complet.dtypes

La base de données contient plus de sept millions d’observations et décrit les licences sportives selon l’année, le département du club sportif, le sport, le sexe, l’âge, et les médailles remportées aux Jeux Olympiques selon les sports. Les types des variables sont cohérents avec leur interprétation. Puisque pandas reconnait les _strings_ comme des _objects_, il est normal que certaines colonnes (comme `sexe` par exemple) soient indiquées être des objets alors qu'elles sont censées être des chaînes de caractères.

#### 2. Valeurs manquantes

In [ ]:
na_table = (
    data_complet.isna()
    .mean()
    .sort_values(ascending=False)
    .to_frame("taux_manquants")
)

na_table

Les valeurs manquantes concernent principalement les données issues de la table `data_medailles`, et sont liées à la jointure par la gauche. Les valeurs manquantes concernant la variable géographique (`code_dep`, soit les départements) sont causées par les clubs sportifs de l'étranger et par les départements non répartis.

In [ ]:
df_code_dep_na = data_complet[data_complet["code_dep"].isna()]
df_code_dep_na["region"].unique()

Après cette vérification, nous pouvons retirer la colonne `region`.

In [ ]:
data_complet = data_complet[
    ["federation",
    "annee",
    "sexe",
    "age",
    "tranche_age",
    "grande_tranche_age",
    "licences_annuelles",
    "code_sport",
    "code_dep",
    "sport",
    "2024_or",
    "2020_or",
    "2016_or",
    "2024_argent",
    "2020_argent",
    "2016_argent",
    "2024_bronze",
    "2020_bronze",
    "2016_bronze",
    "total_medailles_2016",
    "total_medailles_2020",
    "total_medailles_2024"]
]

#### 3. Cohérence des valeurs

##### a. Années observées

In [ ]:
data_complet["annee"].min(), data_complet["annee"].max()

Les années couvertes par la base (2016-2024) sont cohérentes avec le périmètre temporel de l’étude.

##### b. Effectifs de licenciés

In [ ]:
data_complet["licences_annuelles"].describe()

In [ ]:
(data_complet["licences_annuelles"] < 0).sum()

Aucun effectif négatif n’est observé. Les ordres de grandeur des effectifs sont cohérents avec des données de licences sportives.

#### 4. Unicité des observations

In [ ]:
data_complet.duplicated().sum()

Ces doublons stricts représentent environ 0,03% de notre base de données, soit une proportion négligeable. Nous les supprimons afin de garantir l'unicité des observations, sans impact significatif sur les analyses.

In [ ]:
data_complet = data_complet.drop_duplicates()

### D. Export de la base finale

Nous avons ensuite exporté cette table dans le dossier `data_clean`, lui même dans le dossier `data`.

In [ ]:
#gel_parquet(data_complet, "data_complet.parquet")

## II. Visualisations et statistiques descriptives

Nous importons les fonctions nécessaires à la visualisation graphique de notre base de données. Dans cette partie, nous proposons des représentations graphiques interactives, c'est-à-dire qu'il est possible, selon le type de graphique, de choisir le sport que l'on veut voir (soit un sport en particulier, soit tous les sports) ou alors de choisir l'année représentée. 

La plupart du temps, nous commençons par analyser la tendance globale. Si une année ou un sport en particulier sont mentionnés, nous invitons le lecteur à les sélectionner, à l'aide du menu déroulant du widget, afin de pouvoir visualiser le support de l'analyse. 

In [ ]:
from traitement.prep import charger_donnees
from jo.helpers import (
    classement_sports_medailles,
    croissance_licencies_post_jo
)
from jo.widgets import (
    widget_licences_par_sport,
    widgets_carte_licencies,
    widgets_evolution_licences_tranches_fines_age,
    widget_graphique_licences_et_medailles,
    widgets_evolution_licencies,
    widgets_evolution_licences_tranches_grande_age,
    widgets_heatmap_nbr_licencies,
    widgets_camembert_medailles,
    widgets_repartition_fines_tranches_age_par_sport,
    widgets_repartition_grandes_tranches_age_par_sport,
    widgets_licences_par_sexe,
    widgets_part_jeunes,
    widgets_part_femmes
)
from jo.visualisation_plotly import (
    medailles_par_annee
)

In [ ]:
(data_complet, gdf_dep, data_pop) = charger_donnees(
    data_complet_path="data/data_clean/data_complet.parquet",
    geojson_path="data/data_clean/departements.geojson",
    population_path="data/data_clean/population_dept.csv",
)

### A. Exploration "naïve" de la base

Dans cette première partie, nous proposons une analyse générale des grandes tendances se dégageant de la base de données contruite dans la partie précedente. Ainsi, nous ne nous intéressons pas nécessairement aux tendances existant dans la totalité des 33 sports de la base, mais nous en prenons certains en guise d'exemple. Grâce aux graphiques interactifs, le lecteur pourra, par exemple, suivre son sport de prédilection tout au long de notre découverte de la base ou passer en revue la totalité des sports. 

#### 1. Etude des licenciés sportifs en France

La base de données de licences que nous exploitons offre des informations riches sur le profil socio-démographique des licenciés. Nous explorons donc dans cette partie la répartition géographique des licenciés, ainsi que leur répartition selon leur âge et leur sexe, avant de nous intéresser au lien entre les médailles et la pratique d'un sport.

##### a. Nombre et proportion de licenciés sportifs

Nous analysons dans un premier temps la tendance globale des effectifs de licenciés sportifs dans le temps.

In [ ]:
widget_licences_par_sport(data_complet)

La représentation du nombre total de licenciés sportifs en France en fonction du temps nous permet de remarquer une diminution globale marquée du nombre de licenciés sportifs en 2021, et ce dans presque tous les sports (option "Tous sports" du widget) - sauf golf, surf, voile et équitation qui ont connu cette baisse en 2020 et penthatlon moderne et tir qui n'ont presque pas diminué. Cette baisse est probablement liée à la pandémie de Covid 19 et aux mesures sanitaires prises en France, qui ont fortement restreint la pratique sportive. Il nous faudra inclure cet évènement exceptionnel dans nos analyses et modélisations. 


In [ ]:
widgets_carte_licencies(data_complet, gdf_dep, data_pop)

En ce qui concerne la géographie des licenciés, on constate une forte différenciation de la pratique sportive. Par exemple, en observant la représentation géographique des licenciés en voile en 2023 (année pour laquelle nous disposons de données de population fiable d'une faible non répartition géographique des licenciés), on remarque que la pratique de ce sport se concentre en majorité sur les côtes, notamment en Bretagne, en Normandie et dans le Sud-Est de la France bordant la Méditerranée. De même, le rugby est bien plus pratiqué en 2023 dans le Sud-Ouest de la France.

En regardant d'années en années la proportion de personnes pratiquant un sport en étant licencié (widget Sport : "all"), on aperçoit clairement que la pratique sportive s'est répandue dans la population, entre 2016 et 2024. La Bretagne, quelques départements du Sud-Ouest de la France et des Alpes semblent avoir connu une croissance du nombre de leurs habitants pratiquant un sport en club licencié, ce qui est cohérent avec l'augmentation globale du nombre de licenciés après le Covid, dans un contexte où la croissance démographique est faible. 

Assez étonnamment, on observe que l'Ile-de-France fait partie des régions avec une faible proportion de licenciés sportifs, tous sports et toutes années confondues.  

##### b. Age des licenciés sportifs

In [ ]:
widgets_heatmap_nbr_licencies(data_complet)

La représentation du nombre de licenciés selon la tranche d'âge dans le temps montre une chose claire : pratiquer un sport en en tant que licencié est une affaire de jeunes (entre 5 et 19 ans). En effet, les licenciés dans ces tranches d'âges sont entre trois et huit fois plus nombreux que dans les autres tranches. En ce sens, le fait de pratiquer un sport en club licencié serait en grande partie guidé par un effet d'âge. 

Cependant, on peut relever des exceptions selon le sport pratiqué. Par exemple, les licenciés en golf ont en grande majorité plus de 50 ans. Pour visualiser l'hétérogénéité de l'âge des licenciés selon les sports, nous créons un graphique représentant la répartition des licenciés par sport et grande tranche d'âge. 

In [ ]:
widgets_repartition_grandes_tranches_age_par_sport(data_complet)

En regardant cette répartition sur la totalité de la période traitée, on constate le même phénomène global : la plupart des sport sont pratiqués en majorité par des enfants et des jeunes. Cependant, certains sports font exception à cette tendance : le golf, l'haltérophilie, le tir, l'aviron, le badminton et le triathlon sont en majorité pratiqué en club par des adultes ou des seniors. 

Ce graphique permet également de voir que le phénomène de non répartition selon l'âge est relativement faible, et touche certains sports en particulier, comme le canoë-kayak (près de 5%), la natation, la lutte ou la voile (près de 1,5%).

##### c. Sexe des licenciés sportifs

In [ ]:
widgets_licences_par_sexe(data_complet)

La pratique licenciée d'un sport est également différenciée sur le plan du sexe : tout au long de la période considérée, les femmes pratiquent nettement moins de sport avec une licence. En effet, entre 2016 et 2024, l'écart entre les licenciés hommes et femmes est quasiment constant et s'élève à près de quatre millions de licences. On constate également que la non répartition par sexe n'affecte pas ce résultat.

Cependant cette tendance globale cache de grandes disparitées entre les sports : certains, comme le football, sont majoritairement pratiqués par des hommes (10 fois plus d'hommes que de femmes chaque année), alors que d'autres sont majoritairement pratiqués par des femmes, à l'instar de la gymnastique (cinq fois plus de femmes que d'hommes chaque année). Enfin, d'autres disciplines montrent une répartition plus égalitaire de leurs licenciés : il n'y a que 1,12 fois plus de licenciés hommes que de licenciés femmes pratiquant l'athéltisme. 

Ainsi, cette exploration "naïve" de la base de licenciés nous apprend deux faits marquants que nous pourrons retenir pour guider notre analyse de l'influence des médailles olympiques sur les effectifs de licenciés sportifs : 
- la pandémie de Covid 19 a eu un effet global négatif significatif sur la pratique sportive en France, en diminuant d'un tiers le nombre de licenciés sportifs par rapport à 2019, 
- la pratique sportive est socialement différenciée, que ce soit sur le plan géographique, de l'âge ou du sexe ; et ces différences ne vont pas toujours dans le même sens, selon le sport considéré. 

#### 2. Etude des médailles olympiques remportées par la France

Nous nous intéressons dans cette parties aux médailles olympiques seules. Nous choisissons de nous concentrer sur les médailles ayant un code sport qui n'est pas "DIV", c'est à dire aux médailles obtenues dans un sport pour lequel nous disposons des effectifs de licenciés associés. Ces médailles catégorisées comme "DIV" représentent une part négligeable du total de médailles obtenues sur la période (une médaille d'argent de breaking en 2024, sur les 138 obtenues dans le cadre de sports pour lesquels nous disposons des effectifs de licenciés). 

In [ ]:
widgets_camembert_medailles(data_complet)

La France a remporté 138 médailles olympiques au total, en cumulant celles remportées en 2016, en 2021 et en 2024. La répartition de la couleur des médailles remportées est assez équilibrée (34,1% de bronze, 39,9% d'argent et 26,1% d'or), mais reste dominée par les médailles d'argent.  

In [ ]:
medailles_par_annee(data_complet)

L'évolution temporelle du nombre de médailles remportées montre clairement que l'année 2024 a été la plus fructueuse, avec au total 63 médailles remportées. Le nombre de médailles d'or, d'argent et de bronze remportées est nettement supérieur à celui remporté aux éditions précédentes : rès de deux fois moins de médailles ont été remportées lors des Jeux de 2020. 

Ainsi, on constate que la diminution du nombre de licenciés en 2021 est concomitante à un faible nombre de médailles remportées aux Jeux Olympiques de 2020 (qui ont eu lieu en 2021). Cependant, nous pouvons pas affirmer que le moindre nombre de médailles gagnées cette année là a provoqué cette diminution, qui est probablement en grande partie due aux mesures de restriction de la pratique sportive mises en place lors du Covid 19.

### B. Medailles olympiques et licenciés sportifs

Nous essayons ici d'observer plus précisément les évolutions conjointes des médailles remportées aux Jeux Olympiques et du nombre de licenciés.

Dans un premier temps, il s'agit d'observer quels sports ont remporté le plus de médailles olympiques grâce à la fonction `classement_sports_medailles`. Elle propose un classement  décroissant des sports selon le "score pondéré" des médailles obtenues. Ce score permet de représenter synthétiquement le fait que toutes les médailles ne se valent pas : une médaille d'or rapporte plus de prestige et de visbilité au sport qui la remporte qu'une médaille d'argent, qui elle même confère plus de prestige qu'une médaille de bronze. De ce fait, nous attribuons un poids de 3 aux médailles d'or, de 2 à celles d'argent et de 1 à celles de bronze. Le score pondéré est alors une somme pondérée par ce système des médailles obtenues dans le sport considéré. 

In [ ]:
classement_sports_medailles(data_complet, 'all')

Le sport ayant obtenu le plus de médailles est de loin le judo (23 au total) suivi par l'escrime et la natation. Le judo domine à la fois en terme de quantité et de qualité de médailles, puisqu'il présente le plus grand nombre total de médailles et le plus grand score pondéré. Le fait d'introduire un score représentant le prestige des médailles permet de voir que même si le cyclisme a remporté plus de médailles au total (12) que la natation (11), la natation a gagné des médailles plus prestigieuses, lui conférant un score pondéré plus élevé (23) que celui du cyclisme (21).

Dans un second temps, nous analysons l'évolution temporelle du nombre de médailles obtenues : la France remporte-t-elle un nombre prévisible de médailles à chaque édition des JO ? Les Jeux Olympiques de 2020 ayant eu lieu en 2021 en raison de la pandémie de Covid 19, nous avons décidé de définir l'année des médailles des Jeux Olympiques de 2020 en 2021 sur les graphiques.

In [ ]:
widget_graphique_licences_et_medailles(data_complet)

Nous observons que, juste après l'obtention de médailles olympiques, notamment en or, le nombre de licenciés sportifs dans certaines disciplines (volley ball, rugby, handball...) augmente.  

Cependant, on ne peut pas directement affirmer que cette augmentation est due à l'obtention de médailles olympiques. En effet, comme mentionné précédemment, les effectifs de licenciés ont eu tendance à diminuer fortement en 2021 du fait des mesures de restriction de la pratique sportive. De ce fait, lorsque la pratique sportive collective a pu reprendre l'année suivante, les effectifs de licenciés ont naturellement augmenté, sans pour autant que les individus s'inscrivant dans une fédération aient été motivés par le succès des athlètes français aux Jeux Olympiques. On peut tout de même noter que l'augmentation du nombre de licenciés post covid et post Jeux de 2021 a permis de surpasser les effectifs de licenciés avant ces évenements dans certains sports (volley-ball, rugby...).

Nous tentons finalement de voir quels sports pourraient connaître une explosion d'adhésions suite à des Jeux Olympiques, c'est-à-dire quels sport ont connu une plus forte variation positive de leurs effectifs de licenciés suite à l'obtention de médailles olympiques. Connaître cela nous permettra de nous focaliser sur ces sports par la suite, afin d'étudier plus en détail le phénomène, s'il existe. 


Pour ce faire, nous calculons le taux de croissance du nombre de licenciés sportifs entre deux dates. Idéalement, nous devrions calculer ce taux entre l'année précédant les Jeux Olympiques et l'année suivante. Ceci est en partie possible pour 2016, puisque les données de licences de 2016 correspondent à la saison 2015/2016 (inscription pré-Jeux) ou à l'année 2016 (inscription pré-Jeux jusqu'en août, post-Jeux après), et les données de licences de 2017 correspondent en totalité à des inscriptions ayant eu lieu après les Jeux. Pour les Jeux Olympiques de 2024 en revanche, cela n'est pas possible. Finalement, même si nous disposons des données nécessaires pour le faire pour l'édition 2020 (ayant eu lieu en 2021), nous ne calculerons pas ce taux entre 2021 et 2022 : pour tenter de neutraliser la diminution systématique de licenciés due au Covid en 2021, nous calculons ce taux de croissance entre 2019 et 2022. En effet, nous faisons l'hypothèse qu'en l'absence d'un tel choc, le nombre de licenciés en 2021 aurait été similaire à celui de 2019. 

In [ ]:
display(croissance_licencies_post_jo(data_complet, 2016, 1).head())
display(croissance_licencies_post_jo(data_complet, 2019, 3).head())

On peut dans un premier temps remarquer que le taux de croissance entre deux années ne dépasse pas les 20%. Par ailleurs, un seul sport figure parmi les cinq ayant le plus grand taux de croissance sur les deux périodes considérées : le tir. Ainsi, ces sports ont connu une croissance de leur nombre de licenciés relativement importante au moment des Jeux Olympiques et seront donc particulièrement intéressants à analyser plus finement par la suite. 

### C. Evolution du nombre de licenciés : analyse socio-démographique croisée avec le profil des athlètes

On pourrait supposer que le mécanisme d'augmentation du nombre de licenciés suite à une médaille remportée aux Jeux Olympiques repose sur 
l'identification avec l'athlète : le prestige de la médaille donnerait de la visibilité à l'athlète par la médiatisation, et les individus s'identifiant à ce dernier, voulant l'imiter, s'inscriraient l'année suivante dans la même discipline. On pourrait également supposer que l'identification est d'autant plus forte que les nouveaux licenciés sont semblables à l'athlète sur le plan social (même tranche d'âge, même genre...). 

Par ailleurs, la base de données nous permet d'étudier comment cette augmentation a lieu en termes géographiques. En effet, une nouvelle médaille olympique dans un sport pourrait faire augmenter le nombre de licenciés de manière intensive ou extensive. Autrement dit, la pratique d'un sport peut se diffuser en France, ou, au contraire, se concentrer aux endroits où il est déjà pratiqué (notamment si sa pratique repose sur l'environnement naturel, comme la voile qui requiert la présence d'une étendue d'eau). 

Cette dernière partie vise donc à explorer les données dans la perspective de voir si elles corroborent ces hypothèses. Pour ce faire, comme nous n'avons pas de données sur les nouveaux licenciés, mais simplement sur les caractéristiques des licenciés totaux dans chaque sport, nous étudions cela, en le mettant en perspective avec le profil des athlètes olympiques ou les caractéristiques du sport étudié. 

#### 1. Une diffusion ou une concentration de la pratique sportive ? 

Commençons par observer comment les effectifs de licenciés varient suite à l'obtention d'une médaille, sur le plan géographique. Pour cela, nous représentons les taux de croissance des effectifs de licenciés entre deux années dans chaque département, pour chaque sport. Intéressons nous en particulier au canoë-kayak, qui a connu une forte augmentation d'adhésions suite aux Jeux de 2016. Ce sport peut se pratiquer dans les rivières, les fleuves, les lacs ou la mer, autrement dit, presque partout en France. 

In [ ]:
widgets_evolution_licencies(data_complet, gdf_dep)

En observant la carte de l'évolution départementale des licenciés de canoë-kayak entre 2016 et 2017, on constate que l'augmentation des licenciés dans ce sport est géographiquement plus extensive qu'intensive. Si l'on peut voir que le nombre de licenciés diminue dans 42 départements, il est également vrai qu'il augmente dans 53 autres départements, ne serait-ce que très légèrement. Ainsi, sachant que l'évolution globale est positive, on en déduit que de nouvelles personnes s'inscrivent au canoë-kayak un peu partout sur le territoire, et pas seulement à certains endroits.

Nous pouvons étudier la même chose pour le volley-ball entre 2019 et 2022, sport avec le plus fort taux de croissance de ses lienciés sur la période : ce sport connaît également une augmentation de ses licenciés dans toute la France, et ce de manière encore plus marquée que le canoë-kayak, dans la mesure ou 12 départements seulement voient leurs licenciés diminer (contre 84 pour l'augmentation). Cependant, on peut remarquer une augmentation spectaculaire du nombre de licenciés (taux de croissance supérieur à 20%) dans le Cher. 

Il semblerait donc que, quand le nombre de licenciés augmente d'une année sur l'autre (pour quelque raison que ce soit), cette augmentation corresponde plus à une diffusion de la pratique qu'à une concentration de cette dernière. 

#### 2. Les jeunes sont-ils plus inspirés par les athlètes olympiques ? 

Les jeunes, définis comme les individus de moins de 20 ans dans notre base de données, pourraient être plus motivés que les autres par une victoire olympique d'après notre hypothèse, dans la mesure où les athlètes olympiques sont des sportifs jeunes qui ont commencé leur carrière sportive tôt dans leur vie. Nous représentons alors l'évolution de la part des jeunes dans chaque sport.

In [ ]:
widgets_part_jeunes(data_complet)

En observant la part des jeunes (âge < 21 ans) pratiquant un sport en étant licencié, tous sports confondus, on remarque dans un premier temps que cette part est relativement stable, dans la mesure où sur toute la période, elle est contenue entre environ 54 et 57%. Elle augmente très légèrement entre 2016 et 2017, mais diminue ensuite ; pour réaugmenter entre 2021 et 2022. Toutefois, ces augmentations sont faibles et ne permettent pas de conclure à une quelconque identification des jeunes aux athlètes olympiques au niveau global. 

En ce qui concerne le volley-ball, sport ayant connu le plus grand taux de croissance de ses licenciés entre 2019 et 2022, on observe que le taux de jeunes a augmenté de six points entre ces deux dates, ce qui est remarquable. Il semblerait donc que l'hypothèse d'une influence plus importante des victoires olympiques sur la pratique d'un sport en club chez les jeunes ne soit pas vérifiée dans les effectifs globaux, mais qu'elle pourrait l'être dans une certaine mesure dans certains sports, notamment parmi ceux qui ont connu une forte augmentation de leurs effectifs à la suite de Jeux Olympiques.

#### 3. Les médaillées des Jeux sont-elles un exemple de pratique sportive pour les femmes ? 

Afin d'observer le mécanisme d'identification des licenciés au profil des athlètes, on peut s'intéresser au genre des licenciés et des athlètes : est-ce que le fait qu'une femme gagne une médaille aux Jeux Olympiques incite plus les femmes à s'inscrire dans le sport où la médaille a été remportée ? Prenons l'exemple de Clarisse Agbegnenou, qui a gagné une médaille d'or de Judo en 2021, et observons alors l'évolution de la part de femmes pratiquant du judo en club. 

In [ ]:
widgets_part_femmes(data_complet)

Dans un premier temps, la part de femmes pratiquant du judo augmente de six points entre 2016 et 2024. Entre 2019 et 2022, la part de femmes pratiquant le judo en club a augmenté de trois points. Ainsi, si la part de femmes dans ce sport augmente dans le temps, et en particulier sur cette période, c'est probablement que les nouveaux inscrits sont en majorité des femmes (ou alors que plus d'hommes ne renouvellent pas leur licence). 

Même si cette rapide étude de la part des femmes pratiquant du judo se révèle aller dans le sens de notre hypothèse, elle ne nous permet évidemment pas de conclure à l'existence d'un tel mécanisme d'identification des femmes à l'athlète féminine qui inciterait à la pratique sportive. En effet, le taux de croissance de la part des femmes pratiquant le judo augmente de manière régulière sur la période, et l'augmentation de la part de femmes postérieure à la victoire d'une femme aux Jeux n'est donc probablement pas due à l'obtention de cette médaille d'or, mais plus une tendance de long terme. 

Ainsi, les statistiques descriptives ne sont pas suffisantes pour conclure à la véracité de nos hypothèses sur les mécanismes portentiels permettant d'expliquer l'augmentation des effectifs de licenciés à la suite de Jeux Olympiques. Nous tenterons donc, dans une dernière partie, de voir grâce à des modélisations économétriques, si l'obtention de médailles olympiques a réellement un effet positif sur les effectifs de licenciés. 

## III. Modélisation : effet des médailles olympiques sur les licenciés


### A. Objectif

On cherche à mesurer si les médailles obtenues aux Jeux Olympiques (par sport) sont associées à une augmentation du nombre de licences les années suivantes.

Pour répondre à cette question, on mobilise deux approches complémentaires :

1) **Modèle économétrique (principal)** : estimation de l’effet moyen des médailles sur la croissance des licenciés, avec effets fixes sport et effets fixes année, et erreurs robustes clusterisées par sport.

2) **Diagnostics complémentaires (“preuves”)** : modèles prédictifs simples pour quantifier le rôle de l’inertie (niveau passé) vs les médailles.
Attention: Ces diagnostics sont descriptifs/prédictifs, pas causaux.

In [ ]:
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

In [ ]:
# Package model
from model.feature_engineering import build_model_dataset
from model.modele_econo import fit_modele_croissance
from model.preuves import eval_ablation_models, ridge_coefficients, test_medals_incremental

### B. Enrichissement du jeu de données et variables explicatives

À partir des données brutes de licences sportives agrégées au niveau (sport, année), nous construisons un ensemble de variables explicatives visant à caractériser la structure démographique, territoriale et temporelle des licenciés.

L’objectif de cet enrichissement est double :
- améliorer la qualité des analyses descriptives ;
- contrôler certaines dimensions structurelles dans les modèles de modélisation afin d’éviter des biais d’interprétation.

#### 1. Variable dépendante : nombre total de licenciés

La variable centrale du panel est `licences_annuelles`, soit le nombre total de licenciés d’un sport donné pour une année donnée. Il est important de préciser que cette variable ne correspond pas au nombre de licences par catégorie d’âge ou par individu, mais elle correspond à la somme de l’ensemble des licences annuelles observées dans un sport pour une année, toutes catégories confondues. Autrement dit, chaque observation (sport, année) mesure la taille annuelle totale de la fédération considérée.

#### 2. Structure démographique des licenciés

Nous construisons plusieurs variables décrivant la composition des licenciés
pour chaque sport et chaque année :

- **Structure par sexe** :
  - `part_femmes` : part des femmes parmi les licenciés du sport.

- **Structure par âge** :
  - `age_mean`, `age_std` : âge moyen et dispersion de l’âge ;
  - parts par grandes tranches d’âge (jeunes, adultes, seniors), calculées
    à partir des effectifs pondérés par le nombre de licences.

Ces variables permettent de contrôler partiellement l’hétérogénéité des
pratiques sportives selon le profil des pratiquants.

#### 3. Dispersion géographique de la pratique

Afin de prendre en compte l’ancrage territorial des sports, nous construisons plusieurs indicateurs géographiques :

- `nb_departements_actifs` : nombre de départements dans lesquels le sport est
  pratiqué ;
- `herfindahl_dep` : indice de concentration géographique, mesurant si la
  pratique est très concentrée ou au contraire diffuse sur le territoire.

Ces variables permettent de distinguer les sports très localisés de ceux ayant une implantation nationale plus homogène.

#### 4. Variables liées aux Jeux Olympiques

Pour chaque sport, les performances olympiques sont mesurées à partir du nombre total de médailles obtenues lors de l’olympiade de référence.

Chaque année est rattachée à l’édition des Jeux Olympiques la plus proche, afin de refléter l’information effectivement disponible pour les pratiquants et les institutions sportives à cette date.

Cette construction permet d’évaluer empiriquement l’hypothèse selon laquelle les succès olympiques peuvent influencer la dynamique des licences sportives.

In [ ]:
df = pd.read_parquet("data/data_clean/data_complet.parquet")
df_model_full = build_model_dataset(df)
df_model_full.sample(5)

Le code sport DIV regroupe les disciplines non olympiques. Ces disciplines ne sont pas exposées au traitement étudié (les médailles olympiques) mais elles représentent un volume important de licenciés. Leur inclusion introduirait un biais de dilution de l’effet des médailles et violerait l’hypothèse de comparabilité entre unités traitées et non traitées.

Nous excluons donc DIV afin de restreindre l’analyse aux sports potentiellement affectés par les Jeux Olympiques.

In [ ]:
df_model_full = df_model_full[df_model_full["code_sport"] != "DIV"].copy()

#### 5. Vérification de cohérence et de reproductibilité

On vérifie :
- unicité des observations (sport–année) ;
- cohérence de l’assignation `jo_ref` ;
- cohérence de `total_medailles` (constante par sport pour un JO donné).

In [ ]:
# Vérification de lunicité des sports par année
print("Duplicats (code_sport, annee):", df_model_full.duplicated(["code_sport","annee"]).sum())

# total_medailles doit être constant au sein de (sport, jo_ref)
tmp = df_model_full.groupby(["code_sport","jo_ref"])["total_medailles"].nunique().reset_index()
print("Max nunique(total_medailles) par (sport, jo_ref):", tmp["total_medailles"].max())


### C. Modèle économétrique principal : croissance post-JO (panel avec effets fixes)

Nous estimons :

$\Delta \log(1+Lic_{s,t}) = \beta \cdot Med_{s,JO} + \alpha_s + \gamma_t + \varepsilon_{s,t}$

- Variable dépendante : $(\Delta \log(1+Lic_{s,t}))$ ≈ taux de croissance annuel du nombre de licenciés (robuste aux niveaux très différents entre sports).
- $\alpha_s$ : effets fixes sport (popularité structurelle, culture de pratique, taille de base).
- $\gamma_t$ : effets fixes année (chocs communs : Covid, politiques publiques, tendances globales).
- Erreurs standards clusterisées par sport : autorise la corrélation intra-sport dans le temps.

Interprétation : si $\beta$ est petit, $100\beta$ ≈ variation en points de % de la croissance annuelle associée à +1 médaille.


“En présence d’effets fixes année, l’effet identifié correspond à un effet relatif entre sports : à année donnée, les sports plus médaillés connaissent-ils une croissance différente des autres ?”

In [ ]:
results_econo = fit_modele_croissance(
    df_model_full=df_model_full,
    medals_col="total_medailles",
    start_year=2017,
    end_year=2024,
    controls=["part_femmes", "age_mean", "nb_departements_actifs"],  
)

print(results_econo.summary())


In [ ]:
beta = float(results_econo.params["med_last"])
ci = results_econo.conf_int().loc["med_last"]

print(f"beta(médailles) = {beta:.6f}")
print(f"Interprétation: +1 médaille -> ~ {100*beta:.2f}% de croissance annuelle (approx.)")
print(f"IC 95%: [{100*ci[0]:.2f}%, {100*ci[1]:.2f}%]")


#### Discussion économétrique (limites)

Même avec effets fixes sport et année, l’interprétation causale doit rester prudente :
- les médailles peuvent être endogènes (sports structurellement forts / mieux financés → plus de médailles et plus de licenciés) ;
- effets hétérogènes selon les sports (médiatisation, accessibilité de la pratique, structure fédérale) ;
- le modèle mesure un effet moyen à court terme, pas une trajectoire complète.

Ces limites motivent les diagnostics ci-dessous, qui mettent en évidence le rôle central de l’inertie dans les licenciés.

### D. Diagnostics / “preuves” complémentaires

Ces diagnostics ne visent pas à établir une causalité stricte, mais à :
1) quantifier le rôle de l’inertie vs médailles en prédiction hors échantillon ;
2) illustrer que la dynamique des licenciés est très persistante ;
3) vérifier si les médailles apportent une information marginale dans le cadre du modèle économétrique.

#### 1. Preuve 1 — Ablation prédictive (inertie vs médailles)

La première approche consiste à comparer plusieurs modèles prédictifs afin d’évaluer l’apport informationnel marginal des médailles par rapport à l’inertie des licences.
- M0 : inertie seule (`log(lic_{t-1})`)
- M1 : médailles seules
- M2 : inertie + médailles

Le découpage temporel est strictement respecté (train / test), et les modèles incluent des effets fixes sport via des variables indicatrices.

##### a. Méthodologie

La variable cible est définie comme le logarithme du nombre de licences annuelles :
$y_{s,t} = \log(1 + \text{licences}_{s,t})$


Trois spécifications sont estimées à l’aide d’une régression Ridge, avec effets fixes sport (encodés par variables indicatrices) :

- M0 – Inertie seule : lag(1) de la variable cible  
- M1 – Médailles seules : nombre total de médailles obtenues lors des Jeux Olympiques de référence  
- M2 – Inertie + médailles : combinaison des deux précédentes

Le découpage temporel est strictement respecté afin d’éviter toute fuite d’information :
- période d’entraînement : 2017–2020
- période de test : 2022–2024

Les performances sont évaluées à l’aide de :
- $R^2$ sur la cible en log  
- MAE sur la cible en niveau (après transformation inverse)

In [ ]:
ablation = eval_ablation_models(df_model_full)
ablation

**Lecture** :
- Si M0 est déjà très performant, cela indique que les licenciés sont fortement inertiels.
- Si M2 n’améliore que marginalement M0, les médailles apportent peu d’information prédictive additionnelle dans ce cadre.
- La performance de M1 doit être interprétée prudemment (corrélation structurelle possible).


##### b. Résultats

Les résultats obtenus sont résumés dans le tableau ci-dessus. Ils mettent en évidence plusieurs points clés :

- Le modèle M0 (inertie seule) affiche une performance très élevée, avec un $R^2$ proche de 0.99.  
  Cela indique que le nombre de licenciés est fortement inerte : le niveau observé une année est très largement expliqué par celui de l’année précédente.

- Le modèle M1 (médailles seules) affiche également un niveau de performance non négligeable, ce qui suggère l’existence d’une association entre performances olympiques et niveau des licences sportives. Toutefois, cette performance reste inférieure à celle du modèle inertiel et s’accompagne d’une erreur absolue moyenne plus élevée sur la cible en niveau.

- Le modèle M2 (inertie + médailles) n’améliore que marginalement les performances du modèle M0.  
  L’ajout des médailles apporte peu d’information prédictive supplémentaire une fois l’inertie prise en compte.

##### c. Interprétation scientifique

Ces résultats suggèrent que la dynamique des licences sportives est dominée
par des mécanismes d’inertie, ce qui est cohérent avec la nature de la variable
étudiée, correspondant à un stock de pratiquants plutôt qu’à un flux.

La performance du modèle M1 indique que les médailles sont corrélées au niveau des licences, mais cette information apparaît largement redondante avec celle contenue dans l’inertie. Cette corrélation peut refléter des effets structurels, certains sports étant à la fois historiquement populaires et plus performants aux Jeux Olympiques, plutôt qu’un effet prédictif autonome des performances olympiques sur l’évolution future des inscriptions.

Dans ce cadre, les médailles ne constituent pas un signal prédictif dominant lorsqu’on cherche à expliquer ou anticiper la dynamique des licences à court et moyen terme.

##### d. Limites et portée de la preuve

La performance très élevée du modèle M0 invite à interpréter les gains marginaux avec prudence : dans un contexte fortement inertiel, il est mécaniquement difficile pour toute autre variable d’améliorer significativement la prédiction.

Par ailleurs, cette preuve est strictement prédictive et ne permet pas de conclure à une relation causale entre médailles et inscriptions. Une analyse économétrique dédiée, intégrant des effets fixes explicites ou des tests formels, est nécessaire pour approfondir cette question.

Néanmoins, cette première preuve constitue un point d’appui solide pour la suite de l’analyse, en mettant clairement en évidence le rôle central de l’inertie et les limites explicatives des médailles dans un cadre prédictif.

#### 2. Preuve 2 — Poids de l’inertie (coefficients Ridge)

Après l’ablation prédictive (Preuve 1), l’objectif est ici de quantifier le poids
relatif des différentes variables dans la prédiction du niveau de licences.
Pour cela, on estime un modèle Ridge et on analyse les coefficients en valeur
absolue afin d’identifier les composantes les plus informatives.

Pour ce faire, on estime un Ridge sur le train et on examine les coefficients (en valeur absolue).
Si `log_lag1` domine nettement les autres, cela quantifie le rôle central de l’inertie.

In [ ]:
coef = ridge_coefficients(df_model_full)
coef.head(15)

**Lecture** :
- un coefficient très élevé sur `log_lag1` reflète l’inertie : le niveau passé explique la majeure partie du niveau actuel.
- les médailles ont généralement un poids plus faible dans la prédiction du niveau.


##### a. Lecture et interprétation des coefficients Ridge

La figure ci-dessus présente les coefficients estimés par un modèle Ridge entraîné sur la période 2017–2020. Les coefficients sont triés par valeur absolue décroissante, ce qui permet d’identifier les variables les plus informatives dans un cadre prédictif.

Un premier résultat saillant est la domination très nette du coefficient associé à `log_lag1`, c’est-à-dire au logarithme du nombre de licences de l’année précédente. Son coefficient est largement supérieur à celui de toutes les autres variables, ce qui confirme quantitativement le rôle central de l’inertie dans la dynamique des licenciés sportifs.

Les coefficients associés aux variables indicatrices de sport (`code_sport_*`) présentent des amplitudes beaucoup plus faibles. Ils capturent principalement des différences structurelles de niveau entre sports, mais n’expliquent qu’une part marginale de la variation conditionnellement à l’inertie. Leur interprétation doit donc rester descriptive et non causale.

La variable de médailles (`med_last`) et la tendance globale (`trend`) apparaissent avec des coefficients nettement plus faibles que `log_lag1`. Cela suggère que, dans un cadre prédictif, les performances olympiques et la tendance temporelle globale apportent peu d’information supplémentaire une fois la dynamique passée des licences prise en compte.

Ces résultats sont pleinement cohérents avec ceux obtenus dans la *Preuve 1* :
- la très forte performance du modèle basé uniquement sur l’inertie,
- le gain marginal apporté par l’ajout des médailles,
- et la difficulté pour toute autre variable de concurrencer le pouvoir explicatif
  du niveau passé des licenciés.

##### b. Limites et portée de l’analyse

Il est important de souligner que cette analyse repose sur un modèle Ridge à visée prédictive. Les coefficients estimés ne doivent pas être interprétés comme des effets causaux. En particulier, la faiblesse relative du coefficient des médailles ne signifie pas l’absence totale d’effet, mais plutôt que cet effet est dominé par des mécanismes d’inertie et de structure propres aux sports.

Néanmoins, cette seconde preuve apporte un éclairage complémentaire utile : elle quantifie explicitement le poids relatif des différentes composantes du modèle et renforce l’idée que la dynamique des licences sportives est avant tout auto-persistante.

#### 3. Preuve 3 — Les médailles apportent-elles une information marginale dans le modèle économétrique ?

On compare :
- un modèle sans médailles (FE sport + FE année + contrôles)
- un modèle avec médailles

On réalise un test de Wald sur l’hypothèse nulle $H_0 : \beta = 0$ (coefficient médailles nul).


In [ ]:

m0, m1, wald = test_medals_incremental(df_model_full)
print(wald)
print("Coefficient médailles :", m1.params.get("med_last", np.nan))

print("beta medals:", m1.params["med_last"])


**Lecture** :
- Si la p-value est faible, l’effet des médailles est statistiquement différent de zéro, une fois contrôlés les effets fixes.
- Cela renforce l’association conditionnelle mesurée, sans garantir une causalité parfaite.


##### a. Résultats du test économétrique incrémental

La *Preuve 3* vise à évaluer formellement l’apport incrémental des performances olympiques sur la croissance annuelle des licences, en contrôlant pour des effets fixes sport et année ainsi que pour des variables de contrôle disponibles.

La variable dépendante est définie comme la variation du logarithme du nombre de licences annuelles :
$d\log(\text{licences}_{s,t}) = \log(1 + \text{licences}_{s,t}) - \log(1 + \text{licences}_{s,t-1})$

Deux modèles sont estimés :
- M0 : modèle sans variable de médailles,
- M1 : modèle incluant la variable de médailles associées aux Jeux Olympiques de référence (`med_last`).

L’hypothèse nulle testée est :
$H_0 : \beta_{\text{med\_last}} = 0$

Le test de Wald conduit aux résultats suivants :
- statistique de test $\chi^2 \approx 0.20$,
- p-value $\approx 0.65$.

Cette p-value largement supérieure aux seuils usuels (5 % ou 10 %) conduit à ne pas rejeter l’hypothèse nulle. Autrement dit, l’effet des médailles sur la croissance des licences n’est pas statistiquement significatif une fois pris en compte les effets fixes sport et année.

Le coefficient estimé associé aux médailles est par ailleurs très faible en amplitude (environ 0.003), ce qui suggère un impact économique limité, même en l’absence de considération statistique.

##### b. Interprétation et mise en perspective

Ces résultats confirment et renforcent les conclusions tirées des preuves précédentes :

- La *Preuve 1* montrait que l’ajout des médailles n’améliore que marginalement la performance prédictive d’un modèle déjà dominé par l’inertie.
- La *Preuve 2* mettait en évidence le poids écrasant de la variable de lag par rapport aux autres signaux.
- La *Preuve 3* apporte une validation économétrique formelle : les médailles ne présentent pas d’effet incrémental significatif sur la croissance des licences dans le cadre spécifié.

L’ensemble de ces résultats suggère que la dynamique des licences sportives est avant tout gouvernée par des mécanismes d’inertie et de structure propres aux disciplines, tandis que les performances olympiques jouent, au mieux, un rôle secondaire et non systématique.


#### 4. Portée des résultats et limites

Les résultats obtenus permettent de répondre de manière claire à la question posée dans le cadre du projet : aucun effet incrémental significatif des médailles olympiques sur la croissance des licences n’est mis en évidence une fois contrôlée l’inertie et les effets fixes structurels.

Les limites évoquées ci-dessous ne remettent pas en cause cette conclusion, mais en précisent le périmètre d’interprétation.

- Tout d’abord, l’analyse repose sur des données annuelles agrégées au niveau sport–année. Ce choix est cohérent avec la disponibilité des données, mais ne permet pas d’identifier des effets de court terme ou des dynamiques d’anticipation fines autour des événements olympiques.

- Ensuite, le cadre retenu privilégie une approche parcimonieuse et robuste, limitant volontairement la complexité des modèles afin d’éviter des problèmes de sur-paramétrisation et de faible puissance statistique. Des effets hétérogènes selon les sports ne sont donc pas explorés ici.

- Enfin, l’objectif du projet n’est pas d’établir une relation causale stricte, mais d’évaluer empiriquement l’existence d’un lien mesurable et robuste dans un cadre réaliste et reproductible. À ce titre, l’absence d’effet significatif constitue un résultat informatif en soi.

### E. Conclusion de la modélisation

L’ensemble des résultats converge vers un constat central : l’évolution du nombre
de licences sportives est avant tout gouvernée par une forte inertie temporelle,
le niveau observé une année étant largement déterminé par celui de l’année
précédente. Cette dynamique persistante structure l’essentiel des trajectoires
observées au niveau des disciplines.

Dans ce contexte fortement inertiel, l’introduction des performances olympiques
apporte peu d’information supplémentaire. Une fois le passé des licences pris
en compte, les médailles n’améliorent que marginalement la capacité prédictive des
modèles et ne modifient pas substantiellement la dynamique de croissance observée.

Cette conclusion est confirmée par les estimations économétriques en panel : toutes
choses égales par ailleurs, en contrôlant pour des effets fixes sport et année,
l’effet moyen des médailles sur la croissance des licences apparaît faible et
instable, et ne constitue pas un déterminant central de l’évolution des effectifs.

Pris ensemble, ces résultats indiquent que les performances olympiques sont
davantage associées à des dynamiques structurelles propres aux disciplines
(popularité, organisation, tradition) qu’à un effet causal direct et généralisable
sur les inscriptions. L’analyse met ainsi en évidence les limites explicatives des
médailles dans un cadre dynamique dominé par l’inertie.

""""""""""""""""""""""""""""""v2"""""""""""""""""""


Les différents modèles mobilisés au cours de cette phase (approches prédictives,
lecture des coefficients Ridge et estimation économétrique en panel) conduisent à
un résultat convergent. La dynamique du nombre de licences sportives est largement
structurée par une forte inertie temporelle, le niveau observé une année donnée
étant principalement expliqué par celui de l’année précédente.

Une fois cette inertie et les effets fixes pris en compte, les performances
olympiques mesurées par le nombre de médailles n’apparaissent pas comme un
déterminant robuste de la croissance annuelle des licences. Leur contribution
reste marginale dans les modèles prédictifs, leur poids relatif est faible dans la
hiérarchie des variables explicatives, et aucun effet incrémental significatif
n’est mis en évidence dans le cadre économétrique retenu.

Ainsi, la phase de modélisation suggère que les évolutions observées du nombre de
licences relèvent avant tout de dynamiques internes et structurelles propres aux
disciplines, plutôt que d’un effet direct et systématique des succès olympiques.


## Conclusion

Au début de ce projet, nous nous demandions si le fait de remporter des médailles olympiques incite à la pratique sportive.

Nos analyses économétriques nous poussent à conclure que les médailles olympiques n'auraient pas d'effet significatif sur l'évolution du nombre de licenciés sportifs, mais que l'inertie serait la variable la plus explicative.

Pour autant, notre travail présente deux limites majeures, qui nous empêchent d'affirmer que nos résultats sont généralisables. D'une part, nos données sont très restreintes dans le temps : nous n'avons accès aux données de licenciés qu'après deux éditions des Jeux Olympiques, et la pandémie de Covid 19 affectent énormément nos données. Ce choc conjoncturel fausse probablement nos modèles. D'autre part, notre mesure de la pratique sportive est très restreinte : nous étudions le nombre de licenciés sportifs, or la pratique sportive encadrée n'est pas la seule pratique sportive des Français. 

Finalement, nous pensons que notre étude pourrait être complétée par la notion de médiatisation : le fait de remporter des médailles olympiques et d'être confronté à la médiatisation d'un sport olympiques incite-t-il à la pratique sportive ? Les données de Médiamétrie seraient en ce sens les plus pertinentes, mais elles ne sont pas en accès public.